# Прогнозирование задержек доставки Olist

Датасет содержит несколько связанных таблиц с информацией о заказах бразильской платформы электронной коммерции Olist.

Цель этого ноутбука — изучить структуру данных, понять назначение и связи таблиц, а также определить, какие данные можно использовать для прогнозирования задержек доставки.

Начнём с таблицы `orders`, в которой одна строка соответствует одному заказу.


### Импорт библиотек и загрузка данных

In [19]:
import pandas as pd

In [20]:
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv', parse_dates=['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
                                                                         'order_delivered_customer_date', 'order_estimated_delivery_date'])

In [21]:
#выводим общую информацию по таблице
print('shape:', orders.shape)
print('column names:', orders.columns)
print('dtypes:', orders.dtypes)

shape: (99441, 8)
column names: Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')
dtypes: order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


Таблица состоит из 99441 строк и 8 столбцов. Из 8, 5 колонок содержат даты.

In [22]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [23]:
print(orders.duplicated('order_id').sum())

0


`order_id` не имеет дубликатов и является уникальным для каждой строки

In [24]:
print(orders.order_status.value_counts())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


Большинство заказов имеют статус delivered — 96 478 из 99 441. Также присутствуют отменённые, недоступные и незавершённые заказы, поэтому перед созданием целевой переменной потребуется определить, какие статусы включать в выборку.

In [25]:
missing_values = pd.DataFrame({'Количество пропусков' : orders.isna().sum(), 
                               'Доля пропусков' : (orders.isna().mean() * 100).round(2)})
missing_values.index.name = 'Столбец'
missing_values

,Количество пропусков,Доля пропусков
Столбец,,
order_id,0,0.00
customer_id,0,0.00
order_status,0,0.00
order_purchase_timestamp,0,0.00
order_approved_at,160,0.16
order_delivered_carrier_date,1783,1.79
order_delivered_customer_date,2965,2.98
order_estimated_delivery_date,0,0.00


Пропуски встречаются только во временных столбцах. Больше всего их в фактической дате доставки покупателю - 2965 значений. Если вычесть количество заказов статус которых указан как доставленный из общего количества, то получится очень близкое к этому значение - 2963. Необходимо проверить, какими статусами объясняются эти пропуски.

In [26]:
orders.groupby('order_status')['order_delivered_customer_date'].agg(всего_заказов='size',
                                                                   пропусков=lambda x: x.isna().sum(),
                                                                   доля_пропусков_проц=lambda x: (x.isna().mean() * 100).round(2))

,всего_заказов,пропусков,доля_пропусков_проц
order_status,,,
approved,2,2,100.00
canceled,625,619,99.04
created,5,5,100.00
delivered,96478,8,0.01
invoiced,314,314,100.00
processing,301,301,100.00
shipped,1107,1107,100.00
unavailable,609,609,100.00


Почти все пропуски относятся к заказам, не обозначенным как доставленные, что логично, ведь у недоставленного заказа не может быть даты доставки, кроме предполагаемой. При этом 8 доставленных заказов все равно имеют пропуски в дате

In [42]:
dates = orders.drop(columns=['order_id', 'customer_id', 'order_status']).agg(['min', 'max']).T
dates

,min,max
Столбец,,
order_purchase_timestamp,2016-09-04 21:15:19,2018-10-17 17:30:18
order_approved_at,2016-09-15 12:16:38,2018-09-03 17:40:06
order_delivered_carrier_date,2016-10-08 10:34:01,2018-09-11 19:48:28
order_delivered_customer_date,2016-10-11 13:46:32,2018-10-17 13:22:46
order_estimated_delivery_date,2016-09-30 00:00:00,2018-11-12 00:00:00


Проверка минимальных и максимальных дат не выявила необычных значений. Данные охватывают примерно один период — с сентября 2016 года по ноябрь 2018 года. Максимальная ожидаемая дата доставки позже максимальной фактической даты, что логично: для последних заказов дата доставки могла быть запланирована на более поздний срок.
Максимальные даты промежуточных этапов, таких как подтверждение заказа и передача перевозчику, могут быть раньше максимальной даты покупки: последние заказы в датасете могли ещё не пройти эти этапы или соответствующие события не были зафиксированы.

In [65]:
for i in range(dates.shape[1] - 1):
    date_pair = dates.iloc[:, i:i+2].dropna()
    print((date_pair.iloc[:, i] > date_pair.iloc[:, i + 1]).sum())


0


Здесь мы проверяли не происходили ли более ранние этапы позже по времени чем более поздние. Проверка не выявила таких случаев

Таблица orders содержит 99 441 заказ и используется как основа для формирования целевой переменной. Идентификатор order_id уникален, поэтому одна строка соответствует одному заказу.

Проверка дат не выявила нарушений порядка фактических этапов обработки заказа. Для почти всех заказов со статусом delivered доступны фактическая и ожидаемая даты доставки: исключением являются только 8 заказов без фактической даты. Следовательно, для остальных доставленных заказов можно определить, была ли доставка выполнена с задержкой.